In [5]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pandas.io.excel import ExcelWriter

In [6]:
class HTMLTableParser:
    """
    This class is parse have two method.
    It parse HTML as a string and then convert this into Excel table.
    """
    
    def parser_url(self):
        """
        This method parse the HTML as a string and search all tables.
        """ 
        
        response = requests.get('https://dance.vftsarr.ru/reg_module/?mode=reglists&competition_id=97')
        soup = BeautifulSoup(response.content, 'lxml') 
        table = soup.find('table', class_ = 'table')
        return [table, self.parse_html_table(table)]
    
    def parse_html_table(self, table):
        """
        This method create a new table of data from HTML.
        """
        
        """
        First things first, find number of rows and columns. 
        Also, find the columns titles.
        """
        
        n_columns = len(table.find_all('th', scope = 'col')) - 1     # Determine the number of rows in the table.
        n_rows = len(table.find_all('th', scope = 'row'))            # Set the number of columns for table.
        column_names = []

        """
        Handle column names wnen find them.
        """
        
        th_tags = table.find_all('th', scope = 'col')
        if len(th_tags) > 0 and len(column_names) == 0:
            for th in th_tags:
                column_names.append(th.get_text())
        del column_names[0]
        """
        Save column titles.
        """
        
        if len(column_names) > 0 and len(column_names) != n_columns:
            raise Exception("Column titles do not match the number of columns")
            
        columns = column_names if len(column_names) > 0 else range(0, n_columns)
        df = pd.DataFrame(columns = columns, index = range(1, n_rows+1))
        df.index.names = [table.find('th', scope = 'col').text]
        row_marker = 0
        for row in table.find_all('tr'):
            column_marker = 0
            columns = row.find_all('td')
            for column in columns:
                df.iat[row_marker,column_marker] = column.get_text().strip()
                column_marker += 1
            if len(columns) > 0:
                row_marker += 1
                
        """
        Convert to float if possible.
        """
        
        for col in df:
            try:
                df[col] = df[col].astype(float)
            except ValueError:
                pass
            
        df.to_excel('Всероссийские соревнования.xlsx', 'Sheet1')
        return df

In [7]:
hp = HTMLTableParser()
table = hp.parser_url()[1]
table.head(18)

,Дата,Доп.,Категория,Заявок,Оплачено,Свободно
№,,,,,,
1,29.09,III,"Взрослые, Латиноамериканская программа",-,104.0,-
2,29.09,III,"Молодежь, Латиноамериканская программа",-,138.0,-
3,29.09,III,"Юниоры-2, Латиноамериканская программа",-,130.0,-
4,29.09,I юн.,"Юниоры-1, Латиноамериканская программа",-,89.0,-
5,30.09,III,"Взрослые, Европейская программа",-,84.0,-
6,30.09,III,"Молодежь, Европейская программа",-,126.0,-
7,30.09,III,"Юниоры-2, Европейская программа",-,112.0,-
8,30.09,I юн.,"Юниоры-1, Европейская программа",-,90.0,-
9,30.09,III юн.,"Дети-2, Европейская программа (4 танца)",-,69.0,-


In [8]:
pd.read_excel('Всероссийские соревнования.xlsx')

,№,Дата,Доп.,Категория,Заявок,Оплачено,Свободно
0,1,29.09,III,"Взрослые, Латиноамериканская программа",-,104,-
1,2,29.09,III,"Молодежь, Латиноамериканская программа",-,138,-
2,3,29.09,III,"Юниоры-2, Латиноамериканская программа",-,130,-
3,4,29.09,I юн.,"Юниоры-1, Латиноамериканская программа",-,89,-
4,5,30.09,III,"Взрослые, Европейская программа",-,84,-
5,6,30.09,III,"Молодежь, Европейская программа",-,126,-
6,7,30.09,III,"Юниоры-2, Европейская программа",-,112,-
7,8,30.09,I юн.,"Юниоры-1, Европейская программа",-,90,-
8,9,30.09,III юн.,"Дети-2, Европейская программа (4 танца)",-,69,-
9,10,30.09,III юн.,"Дети-1, Европейская программа (3 танца)",-,33,-
